#### 3.1 Loading the Dataset

The raw dataset was loaded from the project data directory for preparation and feature engineering. The dataset is kept unchanged in the raw data directory, while all preparation steps are performed within this notebook.

In [49]:
# Importing the required libraries

import os
import joblib
import pandas as pd
import numpy as np

#### Loading the dataset

In [50]:
# Loading the dataset

df = pd.read_csv("../data/raw/dynamic_pricing.csv")

In [51]:
# Checking dataset dimensions

df.shape

(1000, 10)

In [52]:
# Checking the first few records

df.head()

,Number_of_Riders,Number_of_Drivers,Location_Category,Customer_Loyalty_Status,Number_of_Past_Rides,Average_Ratings,Time_of_Booking,Vehicle_Type,Expected_Ride_Duration,Historical_Cost_of_Ride
0,90,45,Urban,Silver,13,4.47,Night,Premium,90,284.257273
1,58,39,Suburban,Silver,72,4.06,Evening,Economy,43,173.874753
2,42,31,Rural,Silver,0,3.99,Afternoon,Premium,76,329.795469
3,89,28,Rural,Regular,67,4.31,Afternoon,Premium,134,470.201232
4,78,22,Rural,Regular,74,3.77,Afternoon,Economy,149,579.681422


In [53]:
# Checking data types

df.dtypes

Number_of_Riders             int64
Number_of_Drivers            int64
Location_Category              str
Customer_Loyalty_Status        str
Number_of_Past_Rides         int64
Average_Ratings            float64
Time_of_Booking                str
Vehicle_Type                   str
Expected_Ride_Duration       int64
Historical_Cost_of_Ride    float64
dtype: object

## 3.2 Defining the Target Variable

The target variable was identified based on the project's regression objective. `Historical_Cost_of_Ride` represents the historical ride fare that the machine learning models will learn to predict.

In [54]:
# Defining the target variable

target = 'Historical_Cost_of_Ride'

In [55]:
# Checking the target variable

print("Target Variable:", target)

Target Variable: Historical_Cost_of_Ride


#### Separating features and target

In [56]:
# Separating features and target

X = df.drop(columns=[target])
y = df[target]

In [57]:
# Checking feature and target dimensions

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1000, 9)
Target shape: (1000,)


## 3.3 Identifying Feature Types

The predictor variables were divided into numerical and categorical features based on their data types and analytical characteristics. This classification will be used to apply the appropriate preprocessing techniques during model preparation.

In [58]:
# Identifying numerical and categorical features

numerical_features = [
    'Number_of_Riders',
    'Number_of_Drivers',
    'Number_of_Past_Rides',
    'Average_Ratings',
    'Expected_Ride_Duration'
]

categorical_features = [
    'Location_Category',
    'Customer_Loyalty_Status',
    'Time_of_Booking',
    'Vehicle_Type'
]

In [59]:
# Displaying feature groups

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Number_of_Riders', 'Number_of_Drivers', 'Number_of_Past_Rides', 'Average_Ratings', 'Expected_Ride_Duration']

Categorical Features:
['Location_Category', 'Customer_Loyalty_Status', 'Time_of_Booking', 'Vehicle_Type']


#### 3.4 Checking Feature Cardinality

The categorical features were examined to confirm their unique categories before applying encoding. This ensures that the preprocessing strategy is based on the actual structure of the dataset.

In [60]:
# Checking unique categories in categorical features

for col in categorical_features:
    print(f"\n{col}:")
    print(df[col].unique())
    print(f"Number of categories: {df[col].nunique()}")


Location_Category:
<ArrowStringArray>
['Urban', 'Suburban', 'Rural']
Length: 3, dtype: str
Number of categories: 3

Customer_Loyalty_Status:
<ArrowStringArray>
['Silver', 'Regular', 'Gold']
Length: 3, dtype: str
Number of categories: 3

Time_of_Booking:
<ArrowStringArray>
['Night', 'Evening', 'Afternoon', 'Morning']
Length: 4, dtype: str
Number of categories: 4

Vehicle_Type:
<ArrowStringArray>
['Premium', 'Economy']
Length: 2, dtype: str
Number of categories: 2


## 3.5 Feature Engineering

A demand-to-supply feature was engineered to capture the relationship between the number of riders and available drivers. The `Rider_Driver_Ratio` represents the number of riders per available driver and provides an additional measure of potential demand pressure within a ride scenario.

In [61]:
# Creating the rider-to-driver ratio

df['Rider_Driver_Ratio'] = (
    df['Number_of_Riders'] / df['Number_of_Drivers']
)

In [62]:
# Inspecting the engineered feature

df['Rider_Driver_Ratio'].describe()

count    1000.000000
mean        3.235461
std         2.533519
min         1.112360
25%         1.658793
50%         2.357143
75%         3.800000
max        17.600000
Name: Rider_Driver_Ratio, dtype: float64

In [63]:
# Checking for missing and infinite values

print(
    "Missing values:",
    df['Rider_Driver_Ratio'].isnull().sum()
)

print(
    "Infinite values:",
    np.isinf(df['Rider_Driver_Ratio']).sum()
)

Missing values: 0
Infinite values: 0


In [64]:
# Updating the feature dataset after feature engineering

X = df.drop(columns=[target])
y = df[target]

In [65]:
# Checking updated feature and target dimensions

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1000, 10)
Target shape: (1000,)


In [66]:
# Updating numerical features

numerical_features = [
    'Number_of_Riders',
    'Number_of_Drivers',
    'Number_of_Past_Rides',
    'Average_Ratings',
    'Expected_Ride_Duration',
    'Rider_Driver_Ratio'
]

In [67]:
# Displaying updated feature groups

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Number_of_Riders', 'Number_of_Drivers', 'Number_of_Past_Rides', 'Average_Ratings', 'Expected_Ride_Duration', 'Rider_Driver_Ratio']

Categorical Features:
['Location_Category', 'Customer_Loyalty_Status', 'Time_of_Booking', 'Vehicle_Type']


#### 3.6 Train-Test Split

The dataset was divided into training and testing subsets to evaluate model performance on unseen data. The test set was kept separate from subsequent preprocessing and model training to prevent data leakage.

In [68]:
# Splitting the dataset into training and testing sets

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [69]:
# Checking the dimensions of the training and testing sets

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (800, 10)
X_test shape: (200, 10)
y_train shape: (800,)
y_test shape: (200,)


#### 3.7 Preprocessing Strategy

The numerical and categorical features require different preprocessing techniques before they can be used for machine learning. Numerical features will be standardized, while categorical features will be encoded using one-hot encoding.

The preprocessing transformations will be fitted only on the training data and then applied to both the training and testing sets to prevent data leakage.

In [70]:
# Importing preprocessing tools

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [71]:
# Creating the preprocessing pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

Why we're doing this

- StandardScaler → puts numerical features on a comparable scale.
- OneHotEncoder → converts categorical variables into machine-readable binary features.
- handle_unknown='ignore' → prevents errors if an unseen category appears in the test or future data.
- ColumnTransformer → applies the correct transformation to each feature group.

#### 3.8 Applying Preprocessing

The preprocessing transformer was fitted using only the training data to prevent information from the test set from influencing the transformation process. The fitted transformer was then used to transform both the training and testing features.

In [72]:
# Fitting the preprocessing transformer on the training data

X_train_processed = preprocessor.fit_transform(X_train)

In [73]:
# Transforming the testing data

X_test_processed = preprocessor.transform(X_test)

In [74]:
# Checking processed feature dimensions

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Processed X_train shape: (800, 18)
Processed X_test shape: (200, 18)


#### 3.9 Inspecting Processed Feature Names

The transformed feature names were inspected to verify that the numerical and categorical variables were processed as expected and that the resulting feature structure is consistent between the training and testing datasets.

In [75]:
# Extracting processed feature names

processed_feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(processed_feature_names))
print("\nProcessed Features:")
print(processed_feature_names)

Number of processed features: 18

Processed Features:
['num__Number_of_Riders' 'num__Number_of_Drivers'
 'num__Number_of_Past_Rides' 'num__Average_Ratings'
 'num__Expected_Ride_Duration' 'num__Rider_Driver_Ratio'
 'cat__Location_Category_Rural' 'cat__Location_Category_Suburban'
 'cat__Location_Category_Urban' 'cat__Customer_Loyalty_Status_Gold'
 'cat__Customer_Loyalty_Status_Regular'
 'cat__Customer_Loyalty_Status_Silver' 'cat__Time_of_Booking_Afternoon'
 'cat__Time_of_Booking_Evening' 'cat__Time_of_Booking_Morning'
 'cat__Time_of_Booking_Night' 'cat__Vehicle_Type_Economy'
 'cat__Vehicle_Type_Premium']


## 3.10 Converting Processed Data to DataFrames

The processed training and testing feature arrays were converted back into pandas DataFrames using the generated feature names. This preserves feature labels and makes the processed datasets easier to inspect, save, and reuse in subsequent modeling notebooks.

In [76]:
# Converting processed arrays into DataFrames

X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=processed_feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=processed_feature_names,
    index=X_test.index
)

In [77]:
# Checking processed DataFrames

print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

X_train_processed shape: (800, 18)
X_test_processed shape: (200, 18)


In [78]:
# Displaying processed training data

X_train_processed.head()

,num__Number_of_Riders,num__Number_of_Drivers,num__Number_of_Past_Rides,num__Average_Ratings,num__Expected_Ride_Duration,num__Rider_Driver_Ratio,cat__Location_Category_Rural,cat__Location_Category_Suburban,cat__Location_Category_Urban,cat__Customer_Loyalty_Status_Gold,cat__Customer_Loyalty_Status_Regular,cat__Customer_Loyalty_Status_Silver,cat__Time_of_Booking_Afternoon,cat__Time_of_Booking_Evening,cat__Time_of_Booking_Morning,cat__Time_of_Booking_Night,cat__Vehicle_Type_Economy,cat__Vehicle_Type_Premium
29,-0.161036,0.568451,0.098440,-0.350000,0.659185,-0.676366,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
535,-1.700311,-1.007844,0.202107,1.231789,-0.304476,-0.248012,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
695,0.095510,1.093883,1.273329,-0.373261,-0.591523,-0.747758,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
557,-1.101704,-0.219696,0.236662,-1.094371,1.335797,-0.668088,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
836,0.950662,1.934574,0.927773,-0.559354,0.208110,-0.753707,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


#### 3.11 Validating the Preprocessed Data

The processed training and testing datasets were validated to confirm that the numerical features were standardized correctly and that the categorical features were encoded into binary indicators without introducing missing or infinite values.

Missing and infinite values

In [79]:
# Checking for missing and infinite values

print("Missing values in X_train_processed:", X_train_processed.isnull().sum().sum())
print("Missing values in X_test_processed:", X_test_processed.isnull().sum().sum())

print("Infinite values in X_train_processed:", np.isinf(X_train_processed).sum().sum())
print("Infinite values in X_test_processed:", np.isinf(X_test_processed).sum().sum())

Missing values in X_train_processed: 0
Missing values in X_test_processed: 0
Infinite values in X_train_processed: 0
Infinite values in X_test_processed: 0


Validating standardized numerical features

In [80]:
# Identifying processed numerical features

numerical_processed_features = [
    feature for feature in processed_feature_names
    if feature.startswith('num__')
]

In [81]:
# Checking standardized numerical features

numerical_processed_features = [
    feature for feature in processed_feature_names
    if feature.startswith('num__')
]

X_train_processed[numerical_processed_features].describe().T

,count,mean,std,min,25%,50%,75%,max
num__Number_of_Riders,800.0,1.998401e-17,1.000626,-1.743068,-0.845158,0.009995,0.865147,1.677542
num__Number_of_Drivers,800.0,6.661338e-17,1.000626,-1.165473,-0.850214,-0.272239,0.568451,3.248153
num__Number_of_Past_Rides,800.0,8.437695e-17,1.000626,-1.767559,-0.869115,0.063885,0.824107,1.687995
num__Average_Ratings,800.0,-6.039613e-16,1.000626,-1.745695,-0.861755,0.033817,0.813080,1.743544
num__Expected_Ride_Duration,800.0,1.154632e-16,1.000626,-1.821728,-0.796557,0.023579,0.864219,1.663852
num__Rider_Driver_Ratio,800.0,1.998401e-17,1.000626,-0.823963,-0.617787,-0.354437,0.200287,5.453854


Validating one-hot encoded features

In [82]:
# Identifying processed categorical features

categorical_processed_features = [
    feature for feature in processed_feature_names
    if feature.startswith('cat__')
]

print("Number of categorical features:", len(categorical_processed_features))

Number of categorical features: 12


In [83]:
# Checking one-hot encoded categorical values

X_train_processed[categorical_processed_features].stack().unique()

array([0., 1.])

#### 3.12 Saving the Prepared Datasets

The processed training and testing datasets were saved to the processed data directory so that subsequent modeling notebooks can directly use the prepared data without repeating the preprocessing workflow.

In [84]:
# Creating the processed data directory

os.makedirs("../data/processed", exist_ok=True)

In [85]:
# Saving the processed training and testing datasets

X_train_processed.to_csv(
    "../data/processed/X_train.csv",
    index=False
)

X_test_processed.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "../data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

In [86]:
# Verifying saved datasets

print("Processed datasets saved successfully.")

print("\nX_train:", X_train_processed.shape)
print("X_test:", X_test_processed.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Processed datasets saved successfully.

X_train: (800, 18)
X_test: (200, 18)
y_train: (800,)
y_test: (200,)


#### 3.13 Saving the Preprocessor

The fitted preprocessing transformer was saved so that the same scaling and encoding configuration can be reused consistently in subsequent workflows without refitting the transformations.

In [87]:
# Creating the models directory

os.makedirs("../models", exist_ok=True)

In [88]:
# Saving the fitted preprocessor

import joblib

joblib.dump(
    preprocessor,
    "../models/preprocessor.joblib"
)

['../models/preprocessor.joblib']

In [89]:
# Verifying the saved preprocessor

print("Preprocessor saved successfully.")
print("Location: ../models/preprocessor.joblib")

Preprocessor saved successfully.
Location: ../models/preprocessor.joblib


#### Final Preparation Validation

In [90]:
# Final validation of prepared datasets

print("X_train:", X_train_processed.shape)
print("X_test:", X_test_processed.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nProcessed features:", len(processed_feature_names))
print("Preprocessor fitted:", hasattr(preprocessor, "transform"))

X_train: (800, 18)
X_test: (200, 18)
y_train: (800,)
y_test: (200,)

Processed features: 18
Preprocessor fitted: True
